<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/1-2_easy-nl2sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
<h1>Large Language Models Projects</h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>1.2-Create a simple Natural Language to SQL using OpenAI</h2>
    
</div>

by [Pere Martra](https://www.linkedin.com/in/pere-martra/)
<hr>

Models: gpt-3.5-turbo / gpt-4o-mini

Colab Environment: CPU

Keys:
* NL2SQL
* Code Generation
* Prompt Hardening.


# SQL Generator
A sample of how to build a translator from natural language to SQL:

* GPT 35 / gpt-4o-mini
* OpenAI


In [ ]:
## Use updated openai version and add jupyter_bokeh
%pip install -qU openai panel jupyter_bokeh

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


In [ ]:
import panel as pn
from openai import OpenAI

client = OpenAI(api_key=get_secret("OPENAI_API_KEY"))

#model = "gpt-3.5-turbo"
#model = "gpt-4o-mini"
model = "gpt-5.4" ## use new model


In [ ]:
def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content


In [ ]:
## This is on p. 20 of book, but needs to be run before the block below,
## described on p. 17 onwards in the book
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''
    if not prompt:               # ignore empty clicks (no API call, no welcome spam)
        return pn.Column(*panels)

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}."})
    ## Changed for robustness
    context.append({'role':'system', 'content':f"""Return SQL only for valid SQL-related questions.
If the user asks for anything else, do not return SQL.
Instead, reply in plain text:
"Sorry, I'm here only to help with SQL queries." """})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Undate the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)

In [ ]:
context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with
this is your SQL, and after that an SQL that can do what the user request.

Your SQL Database is composed by some tables.
Try to Maintain the SQL order simple.
Just after the SQL add a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order
just answer something nice and simple, maximum 10 words, asking him for something that
can be solved with SQL.
"""} ]

context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr", ## nombre = name in Spanish
      "tipo": "int"       ## tipo = type in Spanish
    },
    {
      "nombre": "name",
      "tipo": "string"
    }
  ]
}
"""
})

context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "string"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "string"
    }
  ]
}
"""
})

## This is on p. 21 in the book
#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Order your data…')
button_conversation = pn.widgets.Button(name="generate SQL")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

In [ ]:
#Sample question: "Return the name of the best paid employee"
dashboard